In [ ]:
#evidence nodes 
evidence_nodes = [
    "Bedroom_Activity",
    "Bathroom_Activity",
    "DiningRoom_Activity",
    "GuestRoom_Activity",
    "Kitchen_Activity",
    "LivingRoom_Activity",
    "LoungeChair_Activity",
    "OtherRoom_Activity",
    "OutsideDoor_Activity",
    "WorkArea_Activity",
    "Hall_Activity",
    "OutsideDoor_Open",
    "OutsideDoor_Close",
    "Time_Of_Day",
    "Day_Of_Week",
    "Is_Night",
    "Total_Activity",
    "Total_Contact_Events"
]

In [ ]:
# behavioural nodes

behavioral_nodes = [
    "Home_Occupancy",
    "Resident_Activity",
    "Resident_Sleep",
    "Leaving_Home",
    "Returning_Home",
    "Unusual_Activity"
]

In [ ]:
# decision and utility nodes 

decision_nodes = ["Smart_Home_Action"]

utility_nodes = ["Homeowner_Utility"]

In [1]:
# define the edges
edges = [
    ("Bedroom_Activity", "Resident_Sleep"),
    ("LoungeChair_Activity", "Resident_Sleep"),
    ("Is_Night", "Resident_Sleep"),

    ("Kitchen_Activity", "Resident_Activity"),
    ("LivingRoom_Activity", "Resident_Activity"),
    ("Bathroom_Activity", "Resident_Activity"),
    ("Bedroom_Activity", "Resident_Activity"),
    ("Total_Activity", "Resident_Activity"),

    ("Total_Activity", "Home_Occupancy"),
    ("OutsideDoor_Activity", "Home_Occupancy"),

    ("OutsideDoor_Open", "Leaving_Home"),
    ("Total_Activity", "Leaving_Home"),

    ("OutsideDoor_Close", "Returning_Home"),
    ("Total_Activity", "Returning_Home"),

    ("Total_Activity", "Unusual_Activity"),
    ("Time_Of_Day", "Unusual_Activity"),

    ("Home_Occupancy", "Smart_Home_Action"),
    ("Resident_Sleep", "Smart_Home_Action"),
    ("Leaving_Home", "Smart_Home_Action"),
    ("Returning_Home", "Smart_Home_Action"),
    ("Unusual_Activity", "Smart_Home_Action"),

    ("Smart_Home_Action", "Homeowner_Utility")
]

In [ ]:
# create the model and add edges

model = DiscreteBayesianNetwork(edges)

print("Nodes:", len(model.nodes()))
print("Edges:", len(model.edges()))

In [ ]:
# add the remaining nodes

all_nodes = (
    evidence_nodes
    + behavioral_nodes
    + decision_nodes
    + utility_nodes
)

model.add_nodes_from(all_nodes)

print("Total nodes:", len(model.nodes()))

In [ ]:
# check if the defined graph is a DAG

print("Is DAG:", nx.is_directed_acyclic_graph(
    nx.DiGraph(model.edges())
))

In [ ]:
train_data = train.drop(columns=["timestamp"], errors="ignore")

In [ ]:
#  train the bayesian net
from pgmpy.estimators import BayesianEstimator

model.fit(
    train_data[evidence_nodes + behavioral_nodes],
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=5
)

In [ ]:
for cpd in model.get_cpds():
    print(cpd)
    print("-" * 50)

In [ ]:
# save the model as pkl file 
import pickle
from pathlib import Path

output = Path("outputs/models")
output.mkdir(parents=True, exist_ok=True)

with open(output / "bayesian_network.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully!")